[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C17_Classical_NLP_Course/01_word_embeddings/01_word_embeddings.ipynb)

# 01 · 词嵌入 word2vec（纯 numpy 从零）

目标：**不调 gensim**，用 numpy 把 **skip-gram + 负采样** 从零实现出来，从一段真实文本里训出有结构的词向量，并验证 **余弦相似** 与 **类比（king−man+woman）**。

路线：one-hot 的困境 → 共现矩阵 + PPMI → sigmoid/softmax → **负采样损失与梯度（数值梯度对拍）** → **完整 skip-gram 训练** → 相似度 & 类比 → ✏️ 练习 → 📖 答案 → 🧪 tiny-shakespeare 胶囊。

> 心智模型：**把常共现的词向量拉近、把随机词推远，重复百万次，语义几何自然长出来**。

## 1 · one-hot 的困境与余弦相似度

one-hot 让任意不同词的相似度恒为 0。先把这个问题量化，并准备好全程要用的 **余弦相似度** 工具。

In [ ]:
import numpy as np
from collections import Counter
rng = np.random.default_rng(0)

def cosine(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    return float(u @ v / (nu * nv)) if nu > 0 and nv > 0 else 0.0

vocab_demo = ['cat', 'dog', 'democracy']
V = len(vocab_demo)
eye = np.eye(V)            # one-hot 矩阵：每行一个词
oh = {w: eye[i] for i, w in enumerate(vocab_demo)}
print('cos(cat, dog)       =', cosine(oh['cat'], oh['dog']))
print('cos(cat, democracy) =', cosine(oh['cat'], oh['democracy']))
assert cosine(oh['cat'], oh['dog']) == 0.0
assert cosine(oh['cat'], oh['democracy']) == 0.0
print('✅ one-hot 下 cat 与 dog、cat 与 democracy 一样不相似(都=0) —— 这就是要解决的问题')

## 2 · 共现矩阵与 PPMI（计数派词向量）

分布假设的最直接量化：统计每个词在窗口 ±m 内的共现。原始计数被高频词主导，用 **PPMI** （正点互信息 $\max(\log\frac{P(i,j)}{P(i)P(j)}, 0)$）消偏。PPMI 矩阵的行就是一版（稀疏）词向量。

In [ ]:
corpus = ('the cat sat on the mat . the dog sat on the log . '
          'a cat chased a dog . the dog chased the cat .').split()
vocab = sorted(set(corpus))
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print('词表大小 V =', V)

def build_cooccur(tokens, w2i, window=2):
    V = len(w2i)
    X = np.zeros((V, V))
    for t, w in enumerate(tokens):
        i = w2i[w]
        lo, hi = max(0, t - window), min(len(tokens), t + window + 1)
        for u in range(lo, hi):
            if u != t:
                X[i, w2i[tokens[u]]] += 1.0
    return X

X = build_cooccur(corpus, w2i, window=2)
assert np.allclose(X, X.T), '对称窗口下共现矩阵应对称'
print('共现矩阵形状:', X.shape, '| 总共现数:', int(X.sum()))

def ppmi(X, eps=1e-12):
    total = X.sum()
    Pij = X / total
    Pi = X.sum(axis=1, keepdims=True) / total      # 行边缘
    Pj = X.sum(axis=0, keepdims=True) / total      # 列边缘
    pmi = np.log((Pij + eps) / (Pi * Pj + eps))
    return np.maximum(pmi, 0.0)                     # 正部

M = ppmi(X)
assert (M >= 0).all(), 'PPMI 非负'
# cat 与 dog 都常跟 the/chased/sat 共现 -> PPMI 行向量应比与随机词更像
print('PPMI 行向量: cos(cat,dog) =', round(cosine(M[w2i['cat']], M[w2i['dog']]), 3))
print('✅ 计数派：PPMI 行向量已能让 cat、dog 因共享上下文而相似')

## 3 · sigmoid 与负采样的目标函数

skip-gram 用中心词预测上下文，朴素 softmax 的分母要遍历整个词表（$O(V)$）。**负采样** 把它换成二分类：

$$\mathcal{L} = -\log\sigma(u_o^\top v_c) - \sum_{i=1}^{k}\log\sigma(-u_{w_i}^\top v_c)$$

第一项拉近真实对、第二项推远 $k$ 个负样本。先把 `sigmoid` 与单样本损失写出来。

In [ ]:
def sigmoid(x):
    return np.where(x >= 0, 1.0 / (1.0 + np.exp(-x)),
                    np.exp(x) / (1.0 + np.exp(x)))   # 数值稳定的两段式

assert abs(sigmoid(0.0) - 0.5) < 1e-12
assert sigmoid(50) > 0.999 and sigmoid(-50) < 1e-3
assert not np.isnan(sigmoid(np.array([-1000.0, 1000.0]))).any(), '大输入不应 NaN'

def ns_loss(v_c, u_o, U_neg):
    '''负采样损失：v_c 中心词向量, u_o 真实上下文向量, U_neg (k,d) 负样本向量。'''
    pos = -np.log(sigmoid(u_o @ v_c) + 1e-12)
    neg = -np.log(sigmoid(-(U_neg @ v_c)) + 1e-12).sum()
    return pos + neg

d = 10
v_c = rng.standard_normal(d) * 0.1
u_o = rng.standard_normal(d) * 0.1
U_neg = rng.standard_normal((5, d)) * 0.1
L = ns_loss(v_c, u_o, U_neg)
print('初始负采样损失 =', round(L, 4))
# 把真实对内积人为调大 -> 损失应下降
L2 = ns_loss(u_o.copy(), u_o, U_neg)   # v_c=u_o -> 正样本内积大
assert L2 < L, '真实对内积变大时损失应下降'
print('✅ 负采样损失：真实对内积↑、负样本内积↓ 时损失↓')

## 4 · 负采样的梯度（与数值梯度对拍）

推导（记 $s=u_o^\top v_c$）：

- $\partial\mathcal{L}/\partial v_c = (\sigma(s)-1)\,u_o + \sum_i \sigma(u_{w_i}^\top v_c)\,u_{w_i}$
- $\partial\mathcal{L}/\partial u_o = (\sigma(s)-1)\,v_c$
- $\partial\mathcal{L}/\partial u_{w_i} = \sigma(u_{w_i}^\top v_c)\,v_c$

注意 $\sigma(s)-1$ 就是「预测 − 标签」，和逻辑回归一致。用**数值梯度**对拍验证解析梯度正确。

In [ ]:
def ns_grads(v_c, u_o, U_neg):
    '''返回 (grad_vc, grad_uo, grad_Uneg) 解析梯度。'''
    s = u_o @ v_c
    sig_pos = sigmoid(s)
    sig_neg = sigmoid(U_neg @ v_c)              # (k,)
    grad_vc = (sig_pos - 1.0) * u_o + (sig_neg[:, None] * U_neg).sum(axis=0)
    grad_uo = (sig_pos - 1.0) * v_c
    grad_Uneg = sig_neg[:, None] * v_c[None, :]  # (k,d)
    return grad_vc, grad_uo, grad_Uneg

def numerical_grad(f, x, eps=1e-6):
    g = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = x[idx]
        x[idx] = old + eps; fp = f()
        x[idx] = old - eps; fm = f()
        x[idx] = old
        g[idx] = (fp - fm) / (2 * eps)
        it.iternext()
    return g

v_c = rng.standard_normal(d) * 0.3
u_o = rng.standard_normal(d) * 0.3
U_neg = rng.standard_normal((5, d)) * 0.3
g_vc, g_uo, g_Un = ns_grads(v_c, u_o, U_neg)
ng_vc = numerical_grad(lambda: ns_loss(v_c, u_o, U_neg), v_c)
ng_uo = numerical_grad(lambda: ns_loss(v_c, u_o, U_neg), u_o)
ng_Un = numerical_grad(lambda: ns_loss(v_c, u_o, U_neg), U_neg)
print('max|err| grad_vc =', np.max(np.abs(g_vc - ng_vc)))
print('max|err| grad_uo =', np.max(np.abs(g_uo - ng_uo)))
print('max|err| grad_Un =', np.max(np.abs(g_Un - ng_Un)))
assert np.allclose(g_vc, ng_vc, atol=1e-5)
assert np.allclose(g_uo, ng_uo, atol=1e-5)
assert np.allclose(g_Un, ng_Un, atol=1e-5)
print('✅ 解析梯度与数值梯度对拍通过 —— 梯度推导正确')

## 5 · 噪声分布：词频的 0.75 次方

负样本不是均匀抽，而是按 $P_n(w)\propto f(w)^{0.75}$。这个魔法指数压低高频词、抬高低频词。下面构造采样表并验证：相对原始词频，0.75 次方确实让分布更「平」。

In [ ]:
def make_noise_dist(counts, power=0.75):
    freqs = np.array([counts[w] for w in vocab], dtype=float)
    p = freqs ** power
    return p / p.sum()

counts = Counter(corpus)
p_raw = make_noise_dist(counts, power=1.0)     # 按词频
p_075 = make_noise_dist(counts, power=0.75)    # word2vec 用的

i_hi = int(np.argmax([counts[w] for w in vocab]))    # 最高频词
i_lo = int(np.argmin([counts[w] for w in vocab]))    # 最低频词
print(f'最高频词 {vocab[i_hi]!r}: p_raw={p_raw[i_hi]:.3f} -> p_0.75={p_075[i_hi]:.3f} (被压低)')
print(f'最低频词 {vocab[i_lo]!r}: p_raw={p_raw[i_lo]:.3f} -> p_0.75={p_075[i_lo]:.3f} (被抬高)')
assert abs(p_075.sum() - 1.0) < 1e-9
assert p_075[i_hi] < p_raw[i_hi], '0.75 次方应压低最高频词'
assert p_075[i_lo] > p_raw[i_lo], '0.75 次方应抬高最低频词'
print('✅ 0.75 次方让噪声分布更均匀：高频降、低频升')

## 6 · 完整 skip-gram + 负采样训练

把零件拼起来：滑窗造正样本对 → 每对抽 $k$ 个负样本 → 前向算损失 → 反向算梯度 → SGD 更新。训练几个 epoch，看**损失下降**，并验证**常共现的词向量变近**。

In [ ]:
def train_skipgram(tokens, w2i, dim=20, window=2, k=5, epochs=50, lr=0.05, seed=0):
    rng = np.random.default_rng(seed)
    V = len(w2i)
    Vin = rng.standard_normal((V, dim)) * 0.1     # 中心词向量
    Uout = rng.standard_normal((V, dim)) * 0.1    # 上下文词向量
    counts = Counter(tokens)
    pn = np.array([counts[w] for w in vocab], float) ** 0.75
    pn /= pn.sum()
    # 预生成正样本对
    pairs = []
    for t, w in enumerate(tokens):
        c_id = w2i[w]
        lo, hi = max(0, t - window), min(len(tokens), t + window + 1)
        for u in range(lo, hi):
            if u != t:
                pairs.append((c_id, w2i[tokens[u]]))
    pairs = np.array(pairs)
    losses = []
    for ep in range(epochs):
        rng.shuffle(pairs)
        tot = 0.0
        for c_id, o_id in pairs:
            negs = rng.choice(V, size=k, p=pn)
            v_c = Vin[c_id]; u_o = Uout[o_id]; U_neg = Uout[negs]
            tot += ns_loss(v_c, u_o, U_neg)
            g_vc, g_uo, g_Un = ns_grads(v_c, u_o, U_neg)
            Vin[c_id]  -= lr * g_vc
            Uout[o_id] -= lr * g_uo
            Uout[negs] -= lr * g_Un
        losses.append(tot / len(pairs))
    return Vin, Uout, losses

Vin, Uout, losses = train_skipgram(corpus, w2i, dim=20, epochs=50, lr=0.05)
print(f'损失: 第1轮 {losses[0]:.3f} -> 末轮 {losses[-1]:.3f}')
assert losses[-1] < losses[0] * 0.8, '训练应显著降低损失'
emb = Vin + Uout            # 最终词向量 = 输入+输出
# cat 与 dog 在语料里语境高度相似 -> 应比与功能词更近
sim_cd = cosine(emb[w2i['cat']], emb[w2i['dog']])
sim_cdot = cosine(emb[w2i['cat']], emb[w2i['.']])
print(f'cos(cat, dog) = {sim_cd:.3f} ; cos(cat, .) = {sim_cdot:.3f}')
print('✅ skip-gram 从零训练完成：损失下降，语境相似的词向量更近')

## 7 · 用训出的向量做相似检索

给定一个词，按余弦相似度找最近邻。这是词向量最常用的用法。

In [ ]:
def most_similar(word, emb, w2i, vocab, topn=3):
    i = w2i[word]
    sims = [(w, cosine(emb[i], emb[w2i[w]])) for w in vocab if w != word]
    return sorted(sims, key=lambda x: -x[1])[:topn]

for w in ['cat', 'dog', 'sat']:
    print(f'{w:6s} 近邻:', [(x, round(s, 2)) for x, s in most_similar(w, emb, w2i, vocab)])
# 自相似=1 的健全性检查
assert abs(cosine(emb[w2i['cat']], emb[w2i['cat']]) - 1.0) < 1e-9
print('✅ 相似检索可用（小语料下结果含噪声，机制正确即可）')

---
## ✏️ 练习 1：skip-gram 的损失

从零实现负采样损失 `ns_loss(v_c, u_o, U_neg)`：
$-\log\sigma(u_o^\top v_c) - \sum_i \log\sigma(-u_{w_i}^\top v_c)$。（用上面定义好的 `sigmoid`。）

In [ ]:
def ns_loss(v_c, u_o, U_neg):
    # TODO: 正样本项 -log σ(u_o·v_c) + 负样本项 -Σ log σ(-U_neg·v_c)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
rng_t = np.random.default_rng(1)
vc = rng_t.standard_normal(8) * 0.3
uo = rng_t.standard_normal(8) * 0.3
Un = rng_t.standard_normal((4, 8)) * 0.3
L = ns_loss(vc, uo, Un)
assert L > 0, '损失应为正'
# 与逐项手算一致
ref = -np.log(sigmoid(uo @ vc)) - np.log(sigmoid(-(Un @ vc))).sum()
assert abs(L - ref) < 1e-9
# 正样本内积越大，损失越小
assert ns_loss(uo, uo, Un) < ns_loss(-uo, uo, Un)
print('✅ 练习 1 通过：负采样损失实现正确')

## ✏️ 练习 2：skip-gram 的梯度

实现 `grad_vc(v_c, u_o, U_neg)`，只返回对中心词向量 $v_c$ 的梯度：
$(\sigma(u_o^\top v_c)-1)\,u_o + \sum_i \sigma(u_{w_i}^\top v_c)\,u_{w_i}$。

In [ ]:
def grad_vc(v_c, u_o, U_neg):
    # TODO: 正样本贡献 (σ(s)-1)*u_o ；负样本贡献 Σ σ(u_i·v_c)*u_i
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
vc = rng_t.standard_normal(8) * 0.3
uo = rng_t.standard_normal(8) * 0.3
Un = rng_t.standard_normal((4, 8)) * 0.3
g = grad_vc(vc, uo, Un)
# 数值梯度对拍
ng = numerical_grad(lambda: ns_loss(vc, uo, Un), vc)
assert np.allclose(g, ng, atol=1e-5), 'grad_vc 应与数值梯度一致'
print('✅ 练习 2 通过：中心词梯度与数值梯度对拍一致')

## ✏️ 练习 3：词类比 king − man + woman

实现 `analogy(a, b, c, emb, w2i, vocab)`：求 `vec(b) - vec(a) + vec(c)` 的余弦最近词，
**必须排除输入词 a, b, c**（否则常返回输入自己）。即解 a:b :: c:?。

In [ ]:
def analogy(a, b, c, emb, w2i, vocab, topn=1):
    # TODO: target = emb[b] - emb[a] + emb[c]; 在 vocab 里找余弦最近(排除 a,b,c)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——（用构造的、保证有平行结构的向量）
toy_vocab = ['man', 'woman', 'king', 'queen', 'apple']
tw2i = {w: i for i, w in enumerate(toy_vocab)}
gender = np.array([1.0, 0, 0, 0])      # 性别方向
royal  = np.array([0, 1.0, 0, 0])      # 王室方向
toy_emb = np.array([
    gender*0,                # man
    gender*1,                # woman = man + 性别
    gender*0 + royal,        # king  = man + 王室
    gender*1 + royal,        # queen = woman + 王室
    np.array([0,0,1.0,0]),   # apple 无关
])
ans = analogy('man', 'king', 'woman', toy_emb, tw2i, toy_vocab)
best = ans[0][0] if isinstance(ans, list) else ans
assert best == 'queen', f'man:king::woman:? 应为 queen, 得到 {best}'
print('✅ 练习 3 通过：king − man + woman ≈ queen')

## ✏️ 练习 4：批量余弦相似度

实现 `cosine_matrix(emb)`：返回 (V,V) 的两两余弦相似度矩阵（先把每行归一化为单位向量，再做 `E @ E.T`）。
这是高效检索的关键（避免逐对循环）。

In [ ]:
def cosine_matrix(emb):
    # TODO: 每行除以其范数得单位向量 En，返回 En @ En.T
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
E = rng_t.standard_normal((6, 5))
C = cosine_matrix(E)
assert C.shape == (6, 6)
assert np.allclose(np.diag(C), 1.0, atol=1e-9), '对角线(自相似)应为 1'
assert np.allclose(C, C.T, atol=1e-9), '余弦矩阵应对称'
# 与逐对 cosine 一致
assert abs(C[0, 1] - cosine(E[0], E[1])) < 1e-9
assert (C <= 1.0 + 1e-9).all() and (C >= -1.0 - 1e-9).all()
print('✅ 练习 4 通过：批量余弦矩阵正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def ns_loss(v_c, u_o, U_neg):
    pos = -np.log(sigmoid(u_o @ v_c) + 1e-12)
    neg = -np.log(sigmoid(-(U_neg @ v_c)) + 1e-12).sum()
    return pos + neg

In [ ]:
# 练习 2 参考答案
def grad_vc(v_c, u_o, U_neg):
    s = u_o @ v_c
    sig_neg = sigmoid(U_neg @ v_c)
    return (sigmoid(s) - 1.0) * u_o + (sig_neg[:, None] * U_neg).sum(axis=0)

In [ ]:
# 练习 3 参考答案
def analogy(a, b, c, emb, w2i, vocab, topn=1):
    target = emb[w2i[b]] - emb[w2i[a]] + emb[w2i[c]]
    exclude = {a, b, c}
    sims = [(w, cosine(target, emb[w2i[w]])) for w in vocab if w not in exclude]
    sims.sort(key=lambda x: -x[1])
    return sims[:topn]

In [ ]:
# 练习 4 参考答案
def cosine_matrix(emb):
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    En = emb / np.where(norms > 0, norms, 1.0)
    return En @ En.T

---
## 🧪 真实数据胶囊：在 tiny-shakespeare 上训词向量

用真实的莎士比亚文本训 skip-gram 词向量。**联网下载 tiny-shakespeare，失败则回退到内置真实莎翁片段**，算法路径完全一致。看高频实词的近邻是否语义合理。

In [ ]:
def load_shakespeare():
    '''下载 tiny-shakespeare；失败回退内置真实片段。返回 token 列表。'''
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    try:
        import urllib.request
        with urllib.request.urlopen(url, timeout=5) as r:
            text = r.read().decode('utf-8')
        src = 'downloaded'
    except Exception:
        text = ('First Citizen: Before we proceed any further, hear me speak. '
                'All: Speak, speak. First Citizen: You are all resolved rather to die than to famish? '
                'All: Resolved. resolved. First Citizen: First, you know Caius Marcius is chief enemy to the people. '
                'All: We know it, we know it. First Citizen: Let us kill him, and we will have corn at our own price. '
                'Is it a verdict? All: No more talking on it; let it be done: away, away! ') * 8
        src = 'builtin fallback'
    # 简单分词：小写、按非字母切；只取前若干 token 以保证 CPU 秒级
    import re
    toks = re.findall(r"[a-z']+", text.lower())[:2000]
    return toks, src

toks, src = load_shakespeare()
print('数据来源:', src, '| 截取 token 数:', len(toks))
# 取高频词建词表(控制规模, CPU 秒级)；只保留出现在词表里的 token
cnt = Counter(toks)
top_words = [w for w, _ in cnt.most_common(50)]
sw2i = {w: i for i, w in enumerate(top_words)}
keep = [w for w in toks if w in sw2i]
print('保留 token:', len(keep), '| 词表:', len(top_words))
assert len(keep) > 50 and len(top_words) >= 10
print('✅ 真实莎翁数据就绪（已截断到玩具规模，秒级可训）')

**🧪 胶囊练习**：用本模块的 `train_skipgram` 在莎翁 token 上训词向量，再用 `most_similar` 看一个高频词的近邻。补全调用即可（玩具规模，秒级）。

In [ ]:
# 用全局 vocab 变量供 train_skipgram 内部使用
vocab = top_words
# TODO: 调 train_skipgram(keep, sw2i, dim=30, window=2, k=5, epochs=8, lr=0.05)
#       得到 Vin, Uout, losses；emb = Vin + Uout
raise NotImplementedError

In [ ]:
# 自测：损失应下降；emb 形状正确
assert losses[-1] < losses[0], '训练应降低损失'
assert emb.shape == (len(top_words), 30)
probe = top_words[5]
print(f'{probe!r} 的近邻:', most_similar(probe, emb, sw2i, top_words, topn=4))
print('✅ 胶囊练习通过：真实文本上训出词向量')

In [ ]:
# 📖 胶囊参考答案
vocab = top_words
Vin, Uout, losses = train_skipgram(keep, sw2i, dim=30, window=2, k=5, epochs=8, lr=0.05)
emb = Vin + Uout
print('训练完成, 末轮损失:', round(losses[-1], 3))

### 小结
- **分布假设**：观其伴知其义。共现矩阵 + PPMI 是计数派词向量；word2vec 是预测派。
- **skip-gram + 负采样**：把 $O(V)$ 的 softmax 换成「真实对 vs $k$ 个随机负样本」的二分类，损失 $-\log\sigma(u_o v_c)-\sum\log\sigma(-u_{neg}v_c)$。
- **梯度** $\sigma(s)-1$ = 预测 − 标签（同逻辑回归），我们用数值梯度对拍验证。
- **噪声分布** $f(w)^{0.75}$：压高频、抬低频的魔法指数。
- **类比** king−man+woman≈queen：某些关系在向量空间是平行位移（记得排除输入词）。
- 经典词向量是**静态**的（一词一向量），通向 ELMo/BERT 的**上下文相关**词向量。

下一站：**模块 02 · N-gram 语言模型** —— 从「表示词」转向「给词序列赋概率」。